In [4]:
from dotenv import load_dotenv
import os
from huggingface_hub import login
import torch

load_dotenv()
hf_token = os.environ["HUGGINGFACE_HUB_TOKEN"]
login(token=hf_token)
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "nightfury2986/llama323-dnd-finetuned"

print(len(hf_token) != 0)

True


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [6]:
import json
with open("chunks.json", "r") as f:
    summarized_chunks = json.load(f)

In [7]:
keys = list(summarized_chunks.keys())

In [8]:
from sentence_transformers import SentenceTransformer
from numpy import ndarray

transformer = SentenceTransformer("all-MiniLM-L6-v2")

In [9]:
embedded_keys = transformer.encode(keys)

In [10]:
import json
import faiss
import numpy as np

# Assume embeddings is a 2D numpy array of shape (num_chunks, dim)
dim = embedded_keys.shape[1]
index = faiss.IndexFlatL2(dim)  # using a simple L2 index
index.add(np.array(embedded_keys))  # add all chunk vectors

def to_text(index):
    key = keys[index]
    chunk = summarized_chunks.get(key)
    return "\n".join(chunk)
    
def rag_query(query):
    query_embedding = transformer.encode([query]) 
    k = 20
    distances, indicesList = index.search(query_embedding, k)
    indicesList = indicesList.tolist()
    return [to_text(index) for indices in indicesList for index in indices]

In [11]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
base_model.eval()
print("Done") # just to avoid the .eval() print

adapter_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/24.4M [00:00<?, ?B/s]

Done


In [12]:
system_prompt = """
# Context
You are an expert in dungeons and dragons. You will be given excerpts from a transcript of a previous dungeons and dragons session
Not all excerpts may be relevant, you must decide which are most relevant and discard the rest
Excerpts will be delimited by === and a newline

# Objective
Answer the user’s question using **only**:
- The provided excerpts
- General Dungeons & Dragons rules or mechanics when needed for clarification

Expand on the response by describing any rules, spells, or mechanics mentioned in the answer

Do not introduce events, facts, or interpretations not supported by the excerpts.
If the answer is not in the excerpts, say 'Not specified in the session'

# Style
Write a short succinct answer.

# Tone
Neutral and matter-of-fact.

# Audience
Someone who participated in or watched the session, and is inquiring about specifics of the session

# Response format
Respond in a few short sentences, with clarifications on rules or mechanics where necessary.
"""

In [13]:
def generate(conversation, model):
    device = model.device
    
    inputs = tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)
    
    input_length = len(inputs[0])
    outputs = model.generate(
        inputs=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=200,
        pad_token_id = tokenizer.eos_token_id
    )
    generated_tokens = outputs[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

def chat(query, model):
    excerpts = rag_query(query)
    conversation = [{
        "role": "system",
        "content": system_prompt
    }, {
        "role": "system",
        "content": "Excerpts:\n" + "\n===\n".join(excerpts)
    }, {
        "role": "user",
        "content": query
    }]
    return generate(conversation, model)

def rule():
    print("\n========================================\n")

def compare(query, verbose=False):
    print(query)
    if verbose:
        print("\n===\n".join(rag_query(query)))
    rule()
    print("Base Model:")
    print(chat(query, base_model))
    #rule()
    #print("Finetuned Model:")
    #print(chat(query, finetuned_model))
    

In [14]:
compare("What city is the party in")

What city is the party in


Base Model:
The party is currently in the city of Red Mountain, which is a coal district.


In [16]:
%%time
compare("Why is the party in the city of Mercedes")

Why is the party in the city of Mercedes


Base Model:
The party is in Mercedes because they were sent there by the Istriots to destroy the city.
CPU times: total: 5min 38s
Wall time: 42.5 s


In [13]:
compare("How does the session start")

How does the session start


Base Model:
The session begins with the DM asking the players if they want to start the adventure immediately or discuss what happened previously. The DM states that they can either "go" or ask what happened first. The players choose to start the adventure.


Finetuned Model:
The session begins with the players discussing how they want to proceed. The DM asks them to recall what happened in the last three weeks, and they explain that they had forgotten their memories and spent time in a trance-like state. They then traveled to Mercedes, where they arrived and flopped down gold to settle their debt.


In [14]:
compare("Why has the party lost their memory")

Why has the party lost their memory


Base Model:
According to the session, the party has lost their memory due to some sort of modification that was done to them. When asked about this, Tim says, "Your memory has been modified. Can you undo it? Maybe with enough time."


Finetuned Model:
Their memory has been modified. Tim can try to undo the modification, but it will take some time. In return, he wants the party to give him telepathic bond and allow him to fly back to the city after three more days.
